# 📗 EDA 시각화 — 데이터 유형별 그래프 선택

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

## 🎯 오늘의 목표
- [ ] **matplotlib 의 Figure / Axes** 구조를 이해하고 그래프에 제목·축 이름을 단다.
- [ ] **분포** 그래프(히스토그램·밀도·상자·바이올린)로 수치 하나의 생김새를 본다.
- [ ] **범주 비교** 그래프(countplot·barplot)로 그룹을 세고 견준다.
- [ ] **관계** 그래프(산점도)로 두 수치가 어떻게 흩어지는지 본다.
- [ ] **시계열 추이**(꺾은선)로 시간에 따른 변화를 읽는다.
- [ ] **집계 히트맵**으로 범주×범주 집계표를 한눈에 본다.
- [ ] **다변량 개괄**(pairplot)로 여러 관계를 한 번에 훑는다.
- [ ] **구성 비율**(파이·누적 막대)과 **보조 차트**(개별 점·누적 분포)를 상황에 맞게 고른다.
- [ ] **그림에서 인사이트**를 뽑고, 그래프를 오독하지 않는다.

## ⏪ 복습 — 지난 시간: 집계로 요약
지난 시간에는 흩어진 표를 **그룹으로 묶어 요약**했습니다.
- `df.groupby('day')['tip'].mean()` — 요일별 평균 팁
- `df.pivot_table(index='day', columns='time', values='tip')` — 요일×시간대 표로 재구성

이렇게 **숫자표**로 요약하면 정확하지만, 표만 봐서는 "어느 쪽이 크고 어떻게 퍼져 있는지"가 한눈에 안 들어옵니다.

## 왜 시각화인가 — 숫자표 vs 그림
같은 데이터라도 **표는 값을 읽는 것**이고 **그림은 형태를 보는 것**입니다.
- 표: `평균 3.26, 2.99, 2.77, 2.73` — 네 숫자를 하나씩 비교
- 그림: 막대 네 개의 **높이**를 한눈에 — 어느 요일이 큰지 즉시 보임

분포의 치우침, 튀는 값, 두 값이 함께 커지는 경향은 **숫자 나열보다 그림에서 훨씬 빨리** 드러납니다. 다음 시간에는 "**어떤 데이터에 어떤 그래프**"를 쓰는지 유형별로 익힙니다.

In [ ]:
# [제공 코드] 시각화 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

---
# 1. matplotlib 기초 — Figure 와 Axes

## 왜 필요할까요?
seaborn 은 예쁜 그래프를 쉽게 그려 주지만, 그 **바탕은 matplotlib** 입니다. 제목·축 이름·여러 그래프 배치는 matplotlib 개념으로 조절하므로 먼저 두 상자를 구분합니다.

- **Figure**: 그림 전체를 담는 **도화지(액자)** 하나.
- **Axes**: 그 안의 **개별 그래프 한 칸** — 축·눈금·제목·데이터가 들어가는 실제 그래프.

한 Figure 안에 Axes 를 여러 칸 둘 수 있습니다(격자 배치).

| 코드 | 하는 일 |
|---|---|
| `fig, ax = plt.subplots(figsize=(8, 4))` | 도화지 1개 + 그래프 칸 1개 만들기 |
| `ax.plot(x, y)` / `ax.bar(...)` | 그 칸에 선·막대 그리기 |
| `ax.set_title('제목')` | 그래프 제목 |
| `ax.set_xlabel('x')` / `ax.set_ylabel('y')` | 축 이름 |
| `plt.figure(figsize=(8, 4))` | 새 도화지 열기(다음 그래프를 여기에) |
| `fig, axes = plt.subplots(1, 2)` | 한 도화지에 그래프 칸 2개(1행 2열) |
| `fig.savefig('output/그림.png')` | 그림을 파일로 저장 |

> seaborn 함수(`sns.histplot` 등)는 그린 **Axes 를 돌려줍니다**. 그래서 `ax = sns.histplot(...)` 로 받아 두고 `ax.set_title(...)` 로 꾸미는 습관을 들이면 좋습니다. 단, 새 그래프를 그리기 전에 `plt.figure()` 로 **새 도화지**를 열어 두면 앞 그림 위에 겹쳐 그려지는 것을 막을 수 있습니다.

In [ ]:
# 오늘 쓸 데이터 세 개를 불러오고, 먼저 살펴봅니다 (head·info)
tips = pd.read_csv('data/restaurant.csv')      # 레스토랑 팁 (분포·범주)
penguins = pd.read_csv('data/penguins.csv')    # 펭귄 측정치 (관계·다변량)
flights = pd.read_csv('data/flights.csv')      # 월별 항공 승객 (시계열)
print('행 수:', len(tips), len(penguins), len(flights))
print('\n[tips 앞부분]'); display(tips.head())
print('\n[tips 구조]'); tips.info()
print('\n[penguins 앞부분]'); display(penguins.head())
# penguins 는 결측이 있습니다 — 관계·다변량 그래프는 결측 행을 조용히 빼고 그립니다(344 -> 333).
print('\n[penguins 구조]'); penguins.info()
# flights 의 month 는 숫자가 아니라 'Jan','Feb' 같은 문자열입니다 — 정렬·축 순서에 영향을 줍니다.
print('\n[flights 구조]'); flights.info()

# 가장 단순한 그래프: 도화지 1개 + 그래프 칸 1개 + 선 하나
year_total = flights.groupby('year')['passengers'].sum()   # 연도별 승객 합계
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(year_total.index, year_total.values, marker='o')
ax.set_title('연도별 항공 승객 합계')
ax.set_xlabel('연도')
ax.set_ylabel('승객 수')
plt.show()

In [ ]:
# 한 도화지에 그래프 칸 두 개 (1행 2열)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 왼쪽 칸(axes[0]): 요일별 평균 팁 — 막대
day_tip = tips.groupby('day')['tip'].mean()
axes[0].bar(day_tip.index, day_tip.values, color='#4C72B0')
axes[0].set_title('요일별 평균 팁')
axes[0].set_ylabel('팁($)')

# 오른쪽 칸(axes[1]): 연도별 승객 합계 — 선
axes[1].plot(year_total.index, year_total.values, marker='o', color='#DD8452')
axes[1].set_title('연도별 승객 합계')
axes[1].set_ylabel('승객 수')

fig.tight_layout()   # 칸들이 겹치지 않게 간격 정리
plt.show()

### 그래프 다듬기 — 범례 · 축 눈금 · 색상

그림을 **읽을 수 있게** 만드는 세 가지 손질입니다.

- **범례(`legend`)** — 선·막대가 여러 개일 때 무엇이 무엇인지 알려 줍니다. `label=` 로 이름을 달고 `ax.legend()` 로 표시합니다.
- **축 눈금** — 라벨이 길어 겹치면 `ax.tick_params(axis='x', rotation=45)` 로 눕힙니다. (교안 안의 갤러리 그림에서 연도 라벨이 겹쳐 보였던 것이 바로 이 문제입니다.)
- **색상 팔레트(`palette=`)** — seaborn 은 색 묶음을 이름으로 바꿀 수 있습니다(`'Set2'`·`'pastel'`·`'muted'` 등). 색은 **구분을 돕는 도구**이지 장식이 아닙니다.

In [ ]:
# 그래프 다듬기 3종 — 범례 · 축 눈금 회전 · 색상 팔레트
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (1) 범례 — 선이 둘 이상이면 label= 로 이름을 달고 legend() 로 표시
axes[0].plot(year_total.index, year_total.values, marker='o', label='연간 합계')
axes[0].plot(year_total.index, year_total.values / 12, marker='s', label='월 평균')
axes[0].set_title('범례 달기')
axes[0].set_ylabel('승객 수')
axes[0].legend(title='구분')

# (2) 축 눈금 회전 — 라벨이 겹칠 때
axes[1].bar(day_tip.index, day_tip.values, color='#4C72B0')
axes[1].set_title('축 눈금 45도 회전')
axes[1].set_ylabel('팁($)')
axes[1].tick_params(axis='x', rotation=45)

fig.tight_layout()
plt.show()

# (3) 색상 팔레트 — 같은 그래프라도 palette= 로 색 묶음을 바꾼다
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=tips, x='day', y='tip', order=['Thur', 'Fri', 'Sat', 'Sun'],
                 errorbar=None, hue='day', palette='Set2', legend=False)
ax.set_title("palette='Set2' 로 색 바꾸기")
plt.show()

In [ ]:
# 완성한 그림을 파일로 저장 — output/ 폴더가 없으면 만든다
import os
os.makedirs('output', exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(tips['total_bill'], bins=20, color='#4C72B0')
ax.set_title('결제 금액 분포')
ax.set_xlabel('total_bill')
fig.savefig('output/total_bill_hist.png', dpi=100, bbox_inches='tight')
print('그림을 output/total_bill_hist.png 로 저장했습니다.')
plt.show()

### 🖐️ 함께 따라하기 — 그래프 칸 두 개 그리기
데모는 **레스토랑 팁·항공 승객** 데이터였죠. 따라하기는 **다른 도메인 — 피트니스 센터 이용 기록**(`gym_visits.csv`)으로 연습합니다. 한 도화지에 그래프를 나란히 두 개 그려 봅시다.

> ⚠️ 이 셀에서 만드는 `gym` 을 **이후 따라하기에서 계속 씁니다** — 건너뛰지 말고 꼭 실행하고 넘어가세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '레스토랑 팁·항공 승객'이었죠. 이번엔 다른 데이터(피트니스 센터 기록)로 연습합니다.
# 1) pd.read_csv 로 data/gym_visits.csv 를 읽어 gym 에 담고 head() 로 열을 확인한다
# 2) plt.subplots 로 1행 2열 그래프 칸을 만든다 (figsize 는 가로로 넓게)
# 3) 왼쪽 칸에는 프로그램별 '운동시간 평균'을 막대(bar)로 그리고 제목을 단다
#    - groupby 로 평균을 먼저 구한 뒤, 그 index 와 values 를 bar 에 넘긴다
# 4) 오른쪽 칸에는 회원등급별 '방문 건수'를 막대로 그리고 제목을 단다
#    - value_counts() 로 개수를 구해 쓴다
# 5) 겹치지 않게 여백을 정리(tight_layout)하고 화면에 표시한다

### ✅ 바로 확인 퀴즈
**1.** Figure 와 Axes 의 차이는 무엇인가요?

<details><summary>정답 보기</summary>

**Figure** 는 그림 전체를 담는 **도화지(액자)** 이고, **Axes** 는 그 안의 **개별 그래프 한 칸**(축·제목·데이터가 들어가는 실제 그래프)입니다. 한 Figure 안에 Axes 를 여러 칸 둘 수 있습니다.

</details>

**2.** 그래프에 제목과 x축 이름을 다는 메서드는 각각 무엇인가요?

<details><summary>정답 보기</summary>

`ax.set_title('제목')` 으로 제목을, `ax.set_xlabel('이름')` 으로 x축 이름을 답니다(y축은 `ax.set_ylabel`). `plt.subplots(1, 2)` 는 Axes 를 두 칸 만들어 `axes[0]`, `axes[1]` 로 씁니다.

</details>

---
## seaborn 이란? — matplotlib 위에 얹는 통계 시각화

방금 본 matplotlib 은 그래프의 **바탕(저수준)** 입니다. **seaborn** 은 그 위에서 동작하는 **고수준 시각화 라이브러리**로, matplotlib 으로는 여러 줄이 필요한 통계 그래프(분포·범주·관계)를 **한 줄 함수 + 컬럼명**으로 그려 줍니다. 그래서 오늘 그래프는 대부분 seaborn 으로 그리고, **제목·축 이름 같은 마무리는 matplotlib(§1)** 으로 합니다.

- **pandas 와 궁합**: `sns.histplot(data=df, x='total_bill')` 처럼 **DataFrame 과 컬럼명**을 그대로 넣습니다. `hue='범주'` 를 주면 그룹별로 색을 자동 구분하고 **범례도 자동 생성**합니다.
- **matplotlib 과의 관계**: seaborn 이 그린 결과도 결국 matplotlib 의 **Axes** 입니다. 그래서 `ax = sns.histplot(...)` 로 받은 뒤 `ax.set_title(...)` 으로 마무리할 수 있습니다.
- **보기 좋은 기본값**: 색·격자·폰트 테마를 `sns.set_theme(style='whitegrid')` 한 줄로 적용합니다(맨 위 [제공 코드] 셀에서 이미 호출했습니다).
- **함수 두 종류**:
  - **Axes 레벨**(대부분 — `histplot`·`boxplot`·`scatterplot`·`barplot`·`lineplot`·`heatmap`): 그래프 **한 칸**에 그리고 그 `Axes` 를 돌려준다.
  - **Figure 레벨**(`pairplot` 등): 여러 칸을 **스스로 만들어** 격자로 그린다.

| 항목 | matplotlib | seaborn |
|---|---|---|
| 위치 | 바탕(저수준) | 그 위 고수준 |
| 입력 | 배열·숫자 값 | DataFrame + 컬럼명 |
| 통계 그래프 | 직접 계산·조립 | 함수 하나로 |
| 그룹 색 구분 | 수동 | `hue=` 로 자동 |

> 한 줄 요약 — **"무엇을 그릴지는 seaborn, 어떻게 꾸밀지는 matplotlib."**

### ✅ 바로 확인 퀴즈
**1.** seaborn 으로 그린 그래프에 **제목**을 달려면 어떻게 하나요?

<details><summary>정답 보기</summary>

seaborn 함수가 돌려준 Axes 를 변수로 받아(`ax = sns.histplot(...)`) matplotlib 의 `ax.set_title('제목')` 으로 답니다. seaborn 은 그림을 그리고, 제목·축 같은 꾸미기는 matplotlib(Axes)으로 마무리합니다.

</details>

**2.** `sns.scatterplot(data=df, x='a', y='b', hue='group')` 에서 `hue='group'` 은 무슨 일을 하나요?

<details><summary>정답 보기</summary>

`group` 열의 값(그룹)별로 점 **색을 자동으로 다르게** 칠하고 **범례도 자동 생성**합니다. matplotlib 으로는 그룹마다 따로 그려야 하는 일을 한 번에 해 줍니다.

</details>

---
# 2. 어떤 데이터에 어떤 그래프? — 시각화 선택 지도

그래프는 **예쁘게** 그리는 게 목적이 아니라 **전하려는 메시지**에 맞게 고르는 것입니다. 핵심 질문 두 가지로 거의 정해집니다.

1. **데이터 유형**이 무엇인가? — 수치형(숫자) / 범주형(그룹) / 시간(순서)
2. **무엇을 전하고 싶은가?** — 분포 / 비교 / 관계 / 추이 / 집계

| 전달 목적 | 데이터 유형 | 권장 차트 | seaborn 함수 |
|---|---|---|---|
| **분포** — 값이 어떻게 퍼져 있나 | 수치형 1개 | 히스토그램·밀도·상자·바이올린 | `histplot` `kdeplot` `boxplot` `violinplot` |
| **비교(개수)** — 그룹별 몇 개인가 | 범주형 1개 | 막대(개수) | `countplot` |
| **비교(대표값)** — 그룹별 평균은 | 범주형 × 수치형 | 막대(평균) | `barplot` |
| **관계** — 두 수치가 함께 움직이나 | 수치형 2개 | 산점도 | `scatterplot` |
| **추이** — 시간에 따라 변하나 | 시간(순서) × 수치형 | 꺾은선 | `lineplot` |
| **집계 격자** — 두 범주의 교차 집계 | 범주형 × 범주형 → 집계값 | 히트맵 | `heatmap`(+`pivot_table`) |
| **다변량 개괄** — 여러 관계를 한 번에 | 수치형 여러 개 | 산점도 행렬 | `pairplot` |
| **구성** — 전체 중 비율 | 범주형(부분/전체) | 파이·누적막대 | matplotlib `pie`·누적 `bar` |
| **보조** — 개별 점·누적·관계+분포 | 상황별 | 점 흩뿌리기·누적분포·산점도+주변분포 | `stripplot` `swarmplot` `ecdfplot` `jointplot` |

## 판단 요령 (빠르게 고르는 규칙)
- 축 하나가 **범주(그룹)** 면 → 막대(`countplot`/`barplot`)나 상자(`boxplot`).
- 축 둘이 모두 **수치** 면 → 산점도(`scatterplot`).
- 축 하나가 **시간(연/월)** 이면 → 꺾은선(`lineplot`).
- **수치 하나**의 생김새만 궁금하면 → 분포 그래프(`histplot`/`kdeplot`/`boxplot`).
- 색(`hue`)·크기(`size`) 를 더하면 그림 하나에 **변수 3~4개**까지 담을 수 있습니다.

> 뒤 섹션에서 이 표의 그래프를 유형별로 하나씩 직접 그려 봅니다. 지금은 "**어떤 상황에 어떤 그래프**"인지 감을 잡는 게 목표입니다.

> 각 유형을 실제 데이터로 그려 본 예시 — 아래 섹션에서 하나씩 배웁니다.

<img src="images/map_gallery.png" width="900"/>

### ✅ 바로 확인 퀴즈
각 상황에 가장 알맞은 그래프(그리고 seaborn 함수)를 골라 보세요.

**1.** 펭귄을 **종(species)별로 몇 마리**씩인지 비교하고 싶다.

<details><summary>정답 보기</summary>

범주별 **개수**이므로 막대그래프 — `countplot`. (평균 같은 대표값이 아니라 행 개수를 세는 상황)

</details>

**2.** **부리 길이**와 **부리 깊이**가 함께 커지는지 보고 싶다.

<details><summary>정답 보기</summary>

두 축이 모두 **수치**이므로 산점도 — `scatterplot`.

</details>

**3.** **연도가 지날수록 승객 수**가 어떻게 변하는지 보고 싶다.

<details><summary>정답 보기</summary>

한 축이 **시간(연도)** 이므로 꺾은선 — `lineplot`.

</details>

---
# 3. 분포 보기 — 수치 하나의 생김새

## 왜 필요할까요?
"평균 20달러" 라는 한 숫자만으로는 값들이 **한쪽으로 쏠렸는지, 넓게 퍼졌는지** 알 수 없습니다. 수치 하나가 어떻게 분포하는지 보는 네 가지 도구입니다.

- **`histplot`** — 값을 구간(**bin**)으로 나눠 각 구간의 **개수**를 막대로. `bins` 를 늘리면 더 잘게, 줄이면 뭉툭.
- **`kdeplot`** — 분포를 **부드러운 밀도곡선**으로. 전체 형태(봉우리 위치)를 매끄럽게 본다.
- **`boxplot`** — 상자=1사분위(Q1)~3사분위(Q3), 가운데 선=**중앙값**, 수염 밖 점=**튀는 값(이상치)**. (이상치는 지난 단원에서 배웠죠)
- **`violinplot`** — 상자그림에 밀도곡선을 씌워 **분포 모양 + 요약**을 한 번에.

> 언제 뭘? — 한 열의 **전체 형태**가 궁금하면 hist/kde, **중앙값·이상치** 같은 요약이 궁금하면 box, **그룹별 분포 비교**엔 box/violin(x에 범주, y에 수치).

> 실제로 그려 본 분포 4종 (같은 데이터, 다른 도구):

<img src="images/dist_4.png" width="820"/>

In [ ]:
# 결제금액(total_bill) 의 분포 — 히스토그램과 밀도곡선
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=tips, x='total_bill', bins=20, ax=axes[0])
axes[0].set_title('결제금액 히스토그램 (bins=20)')
sns.kdeplot(data=tips, x='total_bill', fill=True, ax=axes[1])
axes[1].set_title('결제금액 밀도곡선(KDE)')
plt.show()

In [ ]:
# 요일별 결제금액 분포 비교 — 상자그림 vs 바이올린
order = ['Thur', 'Fri', 'Sat', 'Sun']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=tips, x='day', y='total_bill', order=order, ax=axes[0])
axes[0].set_title('요일별 결제금액 — 상자그림')
sns.violinplot(data=tips, x='day', y='total_bill', order=order, ax=axes[1])
axes[1].set_title('요일별 결제금액 — 바이올린')
plt.show()

### 🖐️ 함께 따라하기 — 소모칼로리 분포 그리기
운동시간 대신 **소모칼로리**의 분포를 히스토그램과 밀도곡선으로 그려 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 새 도화지를 열고(plt.figure), seaborn 의 히스토그램 함수로 gym 의 '소모칼로리' 분포를 그린다
#    - 막대 개수(bins)는 15 정도로 지정해 본다
# 2) 제목을 달고 화면에 표시한다
# 3) 다시 새 도화지를 열고, 밀도곡선 함수(kdeplot)로 같은 열을 그린다 — 곡선 아래를 채워(fill) 본다
# 4) 제목을 달고 표시한다 (그림마다 새 도화지를 여는 것이 겹침 방지의 핵심)

### ✅ 바로 확인 퀴즈
**1.** 상자그림(boxplot)에서 상자의 위·아래 경계와 가운데 선은 각각 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

상자의 아래·위 경계는 **1사분위(Q1)·3사분위(Q3)**, 가운데 선은 **중앙값(Q2)** 입니다. 수염 밖에 찍힌 점은 **튀는 값(이상치)** 후보입니다.

</details>

**2.** 히스토그램에서 `bins` 를 크게 늘리면 그래프가 어떻게 바뀌나요?

<details><summary>정답 보기</summary>

구간이 더 **잘게** 쪼개져 분포를 세밀하게(대신 들쭉날쭉하게) 보여 줍니다. `bins` 를 줄이면 구간이 넓어져 뭉툭해집니다. 밀도곡선(`kdeplot`)은 이를 부드러운 곡선으로 대신 보여 줍니다.

</details>

---
# 4. 비교하기 — 범주별로 세고 견주기

## 왜 필요할까요?
"어느 요일 손님이 많나", "성별로 평균 결제금액이 다른가" 처럼 **그룹(범주)끼리 비교**할 때 막대그래프를 씁니다. 두 막대는 이름이 비슷하지만 **세는 것이 다릅니다**.

- **`countplot`** — 범주별 **행 개수(빈도)**. y축이 자동으로 `count`. ("토요일에 몇 팀 왔나")
- **`barplot`** — 범주별 y값의 **대표값(기본은 평균)**. ("요일별 평균 팁은")
  - **`barplot(..., errorbar=None)` 을 반드시** 넣습니다. 기본 오차막대의 해석은 **다음 단원** 내용이라 지금은 끕니다.
- **`hue='다른범주'`** — 막대를 그룹 안에서 다시 색으로 나눠 **2차원 비교**(요일 × 흡연 여부 등).
- **범례(legend)**: `hue=` 를 주면 각 색이 어떤 그룹인지 알려 주는 **범례가 자동으로 생깁니다**. 위치·제목을 바꾸려면 `ax.legend(title='흡연 여부')` 처럼 조절합니다.

> 헷갈리면: **개수**를 세면 `countplot`, **평균 같은 값**을 견주면 `barplot`.

> 실제로 그려 본 개수(countplot) vs 평균(barplot):

<img src="images/compare_2.png" width="820"/>

In [ ]:
# countplot — 요일별 '방문 건수'(행 개수)
order = ['Thur', 'Fri', 'Sat', 'Sun']
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=tips, x='day', order=order)
ax.set_title('요일별 방문 건수 (countplot = 행 개수)')
plt.show()

In [ ]:
# barplot — 요일별 '평균 팁'(대표값). errorbar=None 필수
order = ['Thur', 'Fri', 'Sat', 'Sun']
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=tips, x='day', y='tip', order=order, errorbar=None)
ax.set_title('요일별 평균 팁 (barplot = 평균)')
plt.show()

# estimator= 로 '무엇을 대표값으로 삼을지' 바꿀 수 있습니다 (기본은 평균).
# 튀는 값이 있을 때는 중앙값(median)이 더 안정적인 대표값입니다.
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=tips, x='day', y='tip', order=order, errorbar=None, estimator='median')
ax.set_title('요일별 팁 중앙값 (estimator=median)')
plt.show()

In [ ]:
# hue 로 그룹 안을 다시 나누기 — 요일 × 흡연 여부별 평균 팁
order = ['Thur', 'Fri', 'Sat', 'Sun']
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=tips, x='day', y='tip', hue='smoker', order=order, errorbar=None)
ax.set_title('요일 × 흡연 여부별 평균 팁')
plt.show()

### 🖐️ 함께 따라하기 — 개수와 평균을 각각 그리기
`countplot` 으로 **개수**를, `barplot` 으로 **평균**을 그려 두 막대의 의미 차이를 느껴 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 새 도화지를 열고, 개수를 세는 seaborn 함수로 '시간대'별 방문 건수를 그린다
# 2) 제목을 달고 표시한다
# 3) 다시 새 도화지를 열고, 평균을 그리는 seaborn 함수로 '프로그램'별 평균 '소모칼로리'를 그린다
#    - 이 함수는 기본으로 오차막대를 그리므로, 오차막대를 끄는 옵션(errorbar)을 반드시 넣는다
# 4) 제목을 달고 표시한다

### ✅ 바로 확인 퀴즈
**1.** `countplot` 과 `barplot` 은 무엇이 다른가요?

<details><summary>정답 보기</summary>

`countplot` 은 범주별 **행 개수(빈도)** 를 그립니다(y축=count). `barplot` 은 범주별 y값의 **대표값(기본 평균)** 을 그립니다. 개수를 세려면 count, 평균을 견주려면 bar 입니다.

</details>

**2.** `barplot` 에 `errorbar=None` 을 넣는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

`barplot` 은 기본으로 막대 위에 **오차막대**를 그리는데, 그 해석은 **다음 단원**에서 배우는 개념입니다. 지금은 평균 막대만 깔끔히 보려고 `errorbar=None` 으로 끕니다.

</details>

---
# 5. 관계 보기 — 두 수치의 흩어짐

## 왜 필요할까요?
"부리가 길면 몸무게도 무거운가?" 처럼 **수치 두 개가 함께 움직이는지** 볼 때 **산점도(`scatterplot`)** 를 씁니다. 점 하나가 표의 한 행이고, 두 축에 두 수치를 놓습니다.

- 점들이 **오른쪽 위로** 늘어서면 → 한쪽이 커질 때 다른 쪽도 커지는 경향.
- **오른쪽 아래로** 늘어서면 → 한쪽이 커질 때 다른 쪽은 작아지는 경향.
- 사방으로 **흩어져** 있으면 → 뚜렷한 관계가 약함.

- **`hue='범주'`** — 점 색으로 그룹(종 등)을 구분.
- **`size='수치'`** — 점 크기로 또 다른 수치를 표현 → 한 그림에 **3~4개 변수**.

> **주의**: 관계의 방향은 지금 **눈으로** 읽습니다. 숫자로 재는 방법은 뒤에서 다루고, 여기서는 그래프 **모양**만 봅니다.

> 실제로 그려 본 산점도 (색=종, 크기=몸무게):

<img src="images/relation_scatter.png" width="560"/>

In [ ]:
# 부리 길이 vs 부리 깊이 — 종(species)별 색으로 구분
plt.figure(figsize=(7, 5))
ax = sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm', hue='species')
ax.set_title('부리 길이 vs 부리 깊이 (종별 색)')
plt.show()

In [ ]:
# 색(종) + 크기(몸무게)까지 — 한 그림에 변수 네 개
plt.figure(figsize=(7, 5))
ax = sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm',
                     hue='species', size='body_mass_g')
ax.set_title('종·몸무게까지 함께 본 산점도')
plt.show()

# style= 는 점의 '모양'을 바꿉니다 — 색과 함께 쓰면 흑백 인쇄나 색약 환경에서도 구분됩니다.
plt.figure(figsize=(7, 5))
ax = sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm',
                     hue='species', style='species')
ax.set_title('색 + 모양(style)으로 이중 구분')
plt.show()

### 🖐️ 함께 따라하기 — 운동시간과 소모칼로리의 관계
오래 운동할수록 칼로리를 더 태울까요? 두 수치의 흩어짐을 산점도로 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 새 도화지를 열고, 산점도 함수로 x='운동시간', y='소모칼로리' 를 그린다
# 2) 프로그램마다 색을 다르게 하려면 색 구분 옵션(hue)에 '프로그램' 을 준다
# 3) 제목을 달고 표시한다
# 4) 점들이 어느 방향으로 늘어서는지, 프로그램별로 기울기가 다른지 눈으로만 읽는다

### ✅ 바로 확인 퀴즈
**1.** 두 수치가 함께 커지는지 보려면 어떤 그래프가 좋을까요?

<details><summary>정답 보기</summary>

**산점도(`scatterplot`)** 입니다. 점이 오른쪽 위로 늘어서면 함께 커지는 경향, 오른쪽 아래면 반대 경향, 사방으로 흩어지면 관계가 약합니다.

</details>

**2.** 산점도에서 `hue` 와 `size` 는 각각 무엇을 표현하나요?

<details><summary>정답 보기</summary>

`hue` 는 점의 **색**으로 범주(예: 종)를 구분하고, `size` 는 점의 **크기**로 또 다른 수치(예: 몸무게)를 표현합니다. 둘을 함께 쓰면 한 그림에 변수 3~4개를 담을 수 있습니다.

</details>

---
# 6. 추이 보기 — 시간에 따른 변화

## 왜 필요할까요?
"승객 수가 해마다 늘고 있나", "여름에 몰리나" 처럼 **시간축에 따른 변화**를 볼 때는 점을 선으로 이은 **꺾은선(`lineplot`)** 이 가장 잘 읽힙니다.

- x축에 **시간(연도·월)**, y축에 수치를 둡니다.
- **`hue='그룹'`** 을 주면 그룹마다 **다른 색 선**을 겹쳐 그려 여러 추이를 한 번에 비교합니다.

> `lineplot` 은 한 x값에 여러 y가 있으면 평균을 내며 오차 띠를 그리는데, 그 해석 역시 다음 단원 개념이라 `errorbar=None` 으로 꺼 두면 선만 깔끔히 보입니다.

> 실제로 그려 본 연도별 추이 (월별 선):

<img src="images/trend_line.png" width="680"/>

In [ ]:
# 연도별 승객 수 추이 (월 평균). errorbar=None 으로 선만 깔끔히
plt.figure(figsize=(8, 4))
ax = sns.lineplot(data=flights, x='year', y='passengers', marker='o', errorbar=None)
ax.set_title('연도별 승객 수 추이 (월 평균)')
plt.show()

In [ ]:
# 여러 선 겹치기 — 1·6·12월의 연도별 추이
sub = flights[flights['month'].isin(['Jan', 'Jun', 'Dec'])]
plt.figure(figsize=(8, 4))
ax = sns.lineplot(data=sub, x='year', y='passengers', hue='month', marker='o')
ax.set_title('월별 승객 추이 (1·6·12월)')
plt.show()

seaborn 의 `lineplot` 에 원본 표를 바로 넣으면 한 x값에 여러 행이 있을 때 **내부에서 평균**을 냅니다. 하지만 이미 `groupby` 로 요약을 끝낸 표라면, `reset_index()` 로 그룹 키를 다시 **열**로 되돌린 뒤(행 하나에 값 하나인 tidy 표) 그 열 이름을 `x=`·`y=` 에 그대로 넣어 그립니다. 또 원본에 `'2024-03-01'` 같은 **시각 문자열**이 들어 있으면, 지난 단원의 `pd.to_datetime` 으로 날짜형으로 바꾸고 `.dt`(연·월·시 등)로 축에 쓸 값을 먼저 만든 다음 같은 방식으로 그립니다.

In [ ]:
# groupby 로 요약한 표를 lineplot 으로 — reset_index() 로 tidy 표를 만든 뒤 x/y 로 넘긴다
year_summary = flights.groupby('year')['passengers'].sum().reset_index()   # 연도별 합계 -> year·passengers 두 열짜리 표
print(year_summary.head())
plt.figure(figsize=(8, 4))                            # 그리기 직전 새 도화지 확보
ax = sns.lineplot(data=year_summary, x='year', y='passengers', marker='o')
ax.set_title('연도별 승객 합계 (집계표를 직접 그리기)')
plt.show()

### 🖐️ 함께 따라하기 — 월별 방문 추이
한 해 동안 방문 건수가 어떻게 오르내리는지 꺾은선으로 봅시다. 날짜에서 **월**을 뽑아 집계한 뒤 그립니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym['방문일'] 을 pd.to_datetime 으로 날짜형으로 바꾼다
# 2) .dt 로 '월'을 뽑아 gym['월'] 열을 만든다
# 3) 월별 방문 건수를 세고(groupby + size), reset_index(name='방문수') 로 tidy 표를 만든다
# 4) 새 도화지를 열고 꺾은선 함수로 x='월', y='방문수' 를 그린다 (점 표시 marker='o', 오차막대는 끈다)
# 5) 제목을 달고 표시한다 — 어느 달에 방문이 몰리는지 확인

### ✅ 바로 확인 퀴즈
**1.** 시간에 따른 변화를 볼 때 가장 알맞은 그래프는 무엇인가요?

<details><summary>정답 보기</summary>

**꺾은선(`lineplot`)** 입니다. x축에 시간(연도·월)을 두고, 점을 선으로 이어 오르내림을 봅니다.

</details>

**2.** 여러 그룹의 추이를 한 그림에 겹쳐 그리려면 어떤 인자를 쓰나요?

<details><summary>정답 보기</summary>

`hue='그룹열'` 을 줍니다. 그룹마다 다른 색 선이 그려져 추이를 한눈에 비교할 수 있습니다.

</details>

---
# 7. 집계를 그림으로 — 히트맵

## 왜 필요할까요?
**두 범주의 교차 집계표**(예: 월 × 연도별 승객 수)는 숫자가 많아 표로는 흐름이 안 보입니다. **히트맵(`heatmap`)** 은 각 칸을 **값의 크기에 따라 색으로 칠해** 큰 곳·작은 곳을 한눈에 보여 줍니다.

- 먼저 앞서 배운 **`pivot_table`** 로 표를 만들고(행=한 범주, 열=다른 범주, 칸=집계값),
- 그 표를 `sns.heatmap(피벗표)` 에 넣습니다.
- **`annot=True`** 면 각 칸에 숫자도 함께 표시, **`cmap`** 으로 색 팔레트를 고릅니다.

> **중요**: 이 히트맵은 **피벗 집계값**(승객 수·평균 팁 등)을 색으로 칠한 것입니다. 숫자가 많은 표를 한눈에 읽는 데 씁니다.

> 실제로 그려 본 피벗 집계 히트맵 (월×연 승객 수):

<img src="images/heatmap_pivot.png" width="680"/>

In [ ]:
# 월 × 연도별 승객 수 — 피벗표를 히트맵으로
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
pv = flights.pivot_table(index='month', columns='year', values='passengers')
pv = pv.reindex(month_order)   # 달 순서를 1~12월로 정렬

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pv, annot=False, cmap='YlGnBu', ax=ax)
ax.set_title('월 × 연도별 승객 수 (피벗 집계값 히트맵)')
plt.show()

### 🖐️ 함께 따라하기 — 요일 × 시간대 히트맵
집계표를 색으로 바꿔 **어느 칸이 큰지** 한눈에 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pivot_table 로 행='요일', 열='시간대', 값='운동시간' 평균 표를 만든다
# 2) plt.subplots 로 도화지와 축을 만든다
# 3) 히트맵 함수로 그 표를 그린다 — 칸 안에 숫자를 표시(annot)하고, 숫자 형식은 소수 1자리로,
#    색 팔레트(cmap)도 하나 골라 지정한다
# 4) 제목을 달고 표시한다 — 평균 운동시간이 가장 긴 칸을 눈으로 찾는다

### ✅ 바로 확인 퀴즈
**1.** 이 히트맵에서 각 **칸의 색**은 무엇을 나타내나요?

<details><summary>정답 보기</summary>

`pivot_table` 로 집계한 **값**(승객 수, 평균 팁 등)입니다. 색이 진할수록 값이 큽니다. 두 범주(행·열)의 교차 **집계표**를 색으로 칠한 것입니다.

</details>

---
# 8. 다변량 개괄과 인사이트

## pairplot — 여러 관계를 한 번에
수치 변수가 여러 개면 둘씩 짝지어 산점도를 그리기 번거롭습니다. **`pairplot`** 은 모든 수치 변수 쌍의 산점도를 **격자로 한 번에** 그려 주고, 대각선에는 각 변수의 분포를 보여 줍니다. `hue='범주'` 로 그룹을 색으로 나눌 수 있어 **탐색 초반 개괄**에 좋습니다.

## 그래프를 오독하지 않기 (체크리스트)
- **파이차트는 범주가 많으면 지양** — 조각이 5개를 넘으면 각도 비교가 어렵습니다. 파이는 **범주 2~4개의 단순 비율**에 쓰고, 정확한 비교엔 **막대**가 낫습니다.
- **막대의 y축은 0에서 시작** — 0이 아닌 곳에서 자르면 작은 차이가 과장돼 보입니다.
- **산점도 과밀 주의** — 점이 너무 겹치면 투명도(`alpha`)를 낮추거나 표본을 줄여 봅니다.
- **제목으로 메시지 전달** — "요일별 평균 팁" 보다 "주말 팁이 평일보다 높다" 처럼 결론을 제목에 담으면 잘 읽힙니다.

## EDA 인사이트 4단계
그래프는 결론이 아니라 **질문에 답하는 과정**입니다.
1. **질문**을 세운다 — "주말이 평일보다 팁을 더 줄까?"
2. **집계**한다 — `groupby`·`pivot_table` 로 숫자 요약.
3. **시각화**한다 — 유형에 맞는 그래프로 그린다.
4. **해석**한다 — 그림에서 읽은 것을 한두 문장으로 정리하고, 다른 설명 가능성도 짚는다.

> 실제로 그려 본 pairplot (펭귄 수치 변수 전체):

<img src="images/pairplot.png" width="680"/>

In [ ]:
# pairplot — 펭귄의 모든 수치 쌍을 한 번에 (종별 색)
# pairplot 은 스스로 새 격자(Figure)를 만들어 plt.figure() 가 필요 없다
grid = sns.pairplot(penguins, hue='species')
plt.show()

### 🖐️ 함께 따라하기 — 변수 세 개만 골라 pairplot
수치 변수 세 개만 골라 더 간결한 산점도 행렬을 그려 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pairplot 에 gym 을 주고, 색 구분(hue)은 '프로그램' 으로 한다
# 2) vars 옵션으로 수치 열 세 개만 고른다 — '운동시간', '소모칼로리', '나이'
# 3) 결과를 변수에 담고 화면에 표시한다 (pairplot 은 새 격자를 스스로 만들어 겹치지 않는다)
# 4) 어느 쌍이 뚜렷한 관계를 보이고, 어느 쌍이 무관해 보이는지 눈으로 읽는다

### 인사이트 도출 예시 (해석 서술)
위 4단계를 팁 데이터에 적용해 보면:

1. **질문**: 주말(토·일)이 평일(목·금)보다 팁을 더 줄까?
2. **집계**: `tips.groupby('day')['tip'].mean()` → 일 3.26, 토 2.99, 목 2.77, 금 2.73
3. **시각화**: 요일별 `barplot(errorbar=None)` — 일·토 막대가 목·금보다 높다.
4. **해석**: "주말의 평균 팁이 평일보다 높아 보인다." 다만 이는 **결제금액 자체가 주말에 크기** 때문일 수 있으니, `tip / total_bill` 비율도 함께 보면 더 정확합니다. — 이렇게 그림 뒤의 **다른 설명 가능성**까지 짚는 것이 좋은 EDA입니다.

In [ ]:
# 위 4단계의 마지막 — '다른 설명 가능성'을 실제로 확인해 봅시다.
# 주말 팁이 높은 게 '인심'인지 '결제금액이 커서'인지, 팁 비율(tip/total_bill)로 갈라 봅니다.
tips['tip_rate'] = tips['tip'] / tips['total_bill']      # 파생 변수 (지난 단원에서 배운 것)

compare = tips.groupby('day')[['tip', 'total_bill', 'tip_rate']].mean().reindex(order)
display(compare.round(3))

print("팁 금액이 가장 큰 요일:", compare['tip'].idxmax())
print("팁 '비율'이 가장 큰 요일:", compare['tip_rate'].idxmax())
# 금액 순위와 비율 순위가 다르다면, '주말에 인심이 후하다'는 결론은 다시 생각해야 합니다.

### ✅ 바로 확인 퀴즈
**1.** 막대그래프의 y축을 0이 아닌 값에서 시작하면 왜 위험한가요?

<details><summary>정답 보기</summary>

막대는 **길이**로 크기를 비교하는데, y축을 0이 아닌 곳에서 자르면 작은 차이가 실제보다 **과장돼** 보입니다. 그래서 막대그래프의 y축은 **0부터** 그리는 것이 정직합니다.

</details>

---
# 9. 구성 보기 — 파이차트와 누적 막대

## 왜 필요할까요?
비교가 "누가 더 큰가"라면, **구성**은 **전체 중 각 부분이 차지하는 비율**을 봅니다.

- **파이차트(`plt.pie` / `ax.pie`)** — 전체를 100%로 놓고 각 조각의 **비율**을 보여 줍니다. 범주가 **2~5개로 적을 때** 직관적입니다.
- **누적 막대(stacked bar)** — 막대 하나를 그룹별로 쌓아 **한 막대 안의 구성비**를 비교합니다. 범주가 많거나 여러 막대를 나란히 견줄 땐 파이보다 낫습니다.

> 파이는 **적은 범주의 단순 비율**에만. 조각이 많아지면 각도 비교가 어려워 **막대**가 더 정확합니다(§8 체크리스트).

<img src="images/composition_2.png" width="820"/>

In [ ]:
# 구성 보기 — 파이차트와 누적 막대 (tips 는 맨 위에서 이미 불러왔습니다)
import numpy as np

day_counts = tips['day'].value_counts().reindex(order)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# 왼쪽: 파이 — 요일 구성비
axes[0].pie(day_counts.values, labels=day_counts.index, autopct='%1.0f%%',
            startangle=90, counterclock=False)
axes[0].set_title('요일 구성비 (파이)')

# 오른쪽: 누적 막대 — 요일별 점심/저녁 비율
ct = pd.crosstab(tips['day'], tips['time']).reindex(order)
ratio = ct.div(ct.sum(axis=1), axis=0)          # 행마다 합이 1이 되게 정규화
bottom = np.zeros(len(ratio))                   # 쌓기 시작 높이 (요일 수만큼 0으로)
for col in ratio.columns:
    axes[1].bar(ratio.index, ratio[col], bottom=bottom, label=col)
    bottom = bottom + ratio[col].values         # 다음 막대는 이만큼 위에서 시작
axes[1].set_title('요일별 점심/저녁 구성비 (누적 막대)')
axes[1].set_ylabel('비율')
axes[1].legend(title='time')
fig.tight_layout()
plt.show()

### 보조 차트 — 개별 점·누적 분포·관계+주변 분포

앞의 8가지로 대부분 해결되지만, 아래 셋은 **특정 상황에서 더 정직하거나 더 많이 보여 줍니다.**

- **`stripplot` · `swarmplot`** — 범주별로 **개별 점을 그대로** 찍습니다. 데이터가 적을 때 상자그림은 요약만 보여 주지만, 이 둘은 **실제 관측치 하나하나**를 보여 줘 더 정직합니다. `swarmplot` 은 점이 겹치지 않게 옆으로 흩어 놓습니다(데이터가 많으면 느리고 답답해집니다).
- **`ecdfplot`** — **누적 분포**입니다. "값이 x 이하인 관측치가 전체의 몇 %인가"를 보여 주며, 히스토그램과 달리 **구간(bins) 개수를 정할 필요가 없어** 왜곡이 없습니다.
- **`jointplot`** — 산점도와 **각 축의 분포**를 한 그림에 함께 보여 줍니다. `pairplot` 처럼 스스로 격자를 만들므로 `plt.figure()` 가 필요 없습니다.


In [ ]:
# 보조 차트 3종 — 상황에 따라 유용한 도구들
# (1) stripplot / swarmplot — 범주별 '개별 점'을 그대로 보여 준다
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.stripplot(data=tips, x='day', y='total_bill', ax=ax[0], alpha=0.6)
ax[0].set_title('stripplot — 점이 겹칠 수 있다')
sns.swarmplot(data=tips, x='day', y='total_bill', ax=ax[1], size=3)
ax[1].set_title('swarmplot — 겹치지 않게 흩어 놓는다')
fig.tight_layout()
plt.show()

# (2) ecdfplot — 누적 분포: 'x 이하가 전체의 몇 %'인지 (구간 나누기가 필요 없다)
plt.figure(figsize=(6, 4))
ax = sns.ecdfplot(data=tips, x='total_bill')
ax.set_title('ecdfplot — 누적 비율로 본 결제금액')
plt.show()

# (3) jointplot — 산점도 + 각 축의 분포를 한 그림에 (스스로 격자를 만든다)
grid = sns.jointplot(data=tips, x='total_bill', y='tip', height=4.5)
plt.show()

### 🖐️ 함께 따라하기 — 파이로 구성비 그리기
전체를 100%로 놓고 **무엇이 얼마를 차지하는지** 파이차트로 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym['회원등급'] 의 value_counts() 로 등급별 개수를 구한다
# 2) plt.subplots 로 도화지와 축을 만든다 (정사각형에 가깝게)
# 3) 축의 pie 로 그 개수를 그린다 — 조각 이름(labels)과 퍼센트 표시(autopct)를 함께 준다
# 4) 제목을 달고 표시한다
# 5) 조각이 몇 개일 때 파이가 읽기 쉬운지 생각해 본다 (많아지면 막대가 낫다)

### ✅ 바로 확인 퀴즈
**1.** 전체 중 각 그룹이 차지하는 **비율**을 보여 줄 때, 범주가 적으면 어떤 그래프가 어울릴까요?

<details><summary>정답 보기</summary>

**파이차트**(또는 누적 막대)입니다. 다만 조각이 5개를 넘으면 각도 비교가 어려워 **막대**가 더 정확합니다.

</details>

**2.** 요일처럼 **여러 막대**에서 각 막대 **안의 구성비**(점심/저녁 비율 등)를 함께 비교하려면 무엇이 좋을까요?

<details><summary>정답 보기</summary>

<strong>누적 막대(stacked bar)</strong>입니다. 파이는 전체 하나의 구성만 보여 주지만, 누적 막대는 여러 그룹의 구성을 **나란히** 비교할 수 있습니다.

</details>

**3.** 데이터가 **적을 때** 상자그림 대신 **관측치 하나하나**를 보여 주고 싶다면? 또 히스토그램처럼 **구간(bins)을 정할 필요 없이** 분포를 보려면?

<details><summary>정답 보기</summary>

개별 점은 <strong>stripplot</strong>(점이 겹칠 수 있음) 또는 <strong>swarmplot</strong>(겹치지 않게 흩어 놓음)입니다. 구간을 정할 필요 없는 분포는 <strong>ecdfplot</strong>(누적 분포)이고, 산점도와 각 축의 분포를 한 번에 보려면 <strong>jointplot</strong> 입니다.

</details>

---
## 🚀 응용 클론코딩 — 한 화면에 담는 미니 대시보드

오늘 배운 그래프들을 **하나의 화면**에 모아 봅시다. 실무 리포트의 첫 장이 대개 이런 모습입니다.

**미션**: 피트니스 데이터로 2×2 대시보드를 만듭니다 — ①운동시간 분포 ②프로그램별 방문 건수 ③운동시간과 소모칼로리의 관계 ④월별 방문 추이. 마지막에 **그림에서 읽은 것을 한 문장**으로 적습니다.

> 각 칸은 서로 다른 **전달 목적**(분포·비교·관계·추이)을 맡습니다. 대시보드는 그 조합이에요.

In [ ]:
# 🖐️ 함께 따라하기 — 미니 대시보드 (아래 순서대로 직접 작성해 보세요)
# 1) plt.subplots 로 2행 2열 격자를 만든다 (figsize 는 넉넉하게, 예: 13×9)
# 2) [0,0] 칸: gym 의 '운동시간' 분포를 히스토그램으로, 제목을 단다
# 3) [0,1] 칸: '프로그램'별 방문 건수를 개수 막대로, 제목을 단다
# 4) [1,0] 칸: x='운동시간', y='소모칼로리' 산점도를 그리고 색은 '프로그램'으로 나눈다
# 5) [1,1] 칸: 앞에서 만든 월별 방문 수(monthly)를 꺾은선으로 그린다
#    (각 seaborn 함수에 ax=axes[행, 열] 을 넘겨 그 칸에 그린다)
# 6) 전체 제목(fig.suptitle)을 달고 tight_layout 으로 여백을 정리한 뒤 표시한다
# 7) 아래 서술 셀에 '이 대시보드에서 읽은 것' 한 문장을 적는다

**이 대시보드에서 읽은 것 (서술)**

*(예시)* 운동시간은 40~70분에 몰려 있고, 방문은 웨이트가 가장 많다. 운동시간이 길수록 소모칼로리가 늘어나되 **프로그램마다 기울기가 달라** 강도 차이가 보인다. 방문은 1월에 크게 몰렸다가 봄에 저점을 찍고 여름에 한 번 반등한다.

> 대시보드는 "무엇이 보이는가"까지가 절반이고, **"그래서 무엇을 할 것인가"**를 적는 순간 리포트가 됩니다.

---
## 이번 강의 정리 — 전달 목적별 차트 지도

| 전달 목적 | 추천 차트 | seaborn 함수 | 언제 쓰나 |
|---|---|---|---|
| 분포 보기 | 히스토그램·밀도·상자·바이올린 | `histplot` `kdeplot` `boxplot` `violinplot` | 수치 1개의 생김새 |
| 개수 비교 | 막대(개수) | `countplot` | 범주별 빈도 |
| 대표값 비교 | 막대(평균) | `barplot`(`errorbar=None`) | 범주별 평균 |
| 관계 보기 | 산점도 | `scatterplot` | 수치 2개 |
| 추이 보기 | 꺾은선 | `lineplot` | 시간축 |
| 집계 격자 | 히트맵 | `heatmap`(+`pivot_table`) | 범주 × 범주 집계 |
| 다변량 개괄 | 산점도 행렬 | `pairplot` | 여러 수치 한 번에 |
| 구성 보기 | 파이·누적막대 | matplotlib `pie`·누적 `bar` | 부분→전체 비율 |
| 보조 차트 | 점 흩뿌리기·누적분포·결합 | `stripplot` `swarmplot` `ecdfplot` `jointplot` | 개별 점·누적 비율·관계+분포 |

이제 여러분은 데이터 유형과 전하려는 메시지에 맞춰 **알맞은 그래프를 골라** 그릴 수 있습니다.

## ⏭️ 예고 — 다음 단원: 기술통계·확률
지금까지 **그림으로** 본 분포·관계를 이제 **수치로** 요약합니다.
- **기술통계** — 평균·분산·표준편차 등으로 분포를 숫자로 요약
- **확률의 기초** — 데이터를 확률의 눈으로 보기

오늘 익힌 "유형별 그래프 선택"이 그 모든 분석의 **첫 관문**입니다. 수고하셨습니다!